In [7]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### 3.- SERIES DE TIEMPO

#### 3-3 Zonas Horarias

Horario central del México tiene 6 horas de diferencia respecto al meridiano de greenwich (Londres)(UTC) 

Nota:

Cual es la diferencia entre el tipo de datos datetime y timestamp en mysql? 

En MySQL, tanto el tipo de dato DATETIME como TIMESTAMP se utilizan para almacenar fechas y horas, pero hay algunas diferencias importantes entre ellos:

DATETIME:
- Almacena fechas y horas en un formato específico (AAAA-MM-DD HH:MM:SS)
- No tiene zona horaria asociada
- No se actualiza automáticamente cuando se inserta o actualiza un registro
- Puede almacenar fechas y horas en cualquier rango (desde 1000-01-01 00:00:00 hasta 9999-12-31 23:59:59)
- Requiere 8 bytes de almacenamiento

TIMESTAMP:
- Almacena fechas y horas en un formato específico (AAAA-MM-DD HH:MM:SS)
- Tiene zona horaria asociada (UTC por defecto, es decir, el meridiano de greenwich)
- Se actualiza automáticamente cuando se inserta o actualiza un registro (si se configura para hacerlo)
- Puede almacenar fechas y horas en un rango más limitado (desde 1970-01-01 00:00:00 hasta 2038-01-19 03:14:07)
- Requiere 4 bytes de almacenamiento

En resumen, la principal diferencia entre DATETIME y TIMESTAMP es que TIMESTAMP tiene zona horaria asociada y se puede configurar para actualizarse 
automáticamente, mientras que DATETIME no tiene zona horaria asociada y no se actualiza automáticamente.

Según la experiencia y las prácticas comunes en la industria, el tipo de dato más utilizado en MySQL es DATETIME.
Hay varias razones por las que DATETIME es más popular:
1. Flexibilidad: DATETIME permite almacenar fechas y horas en cualquier rango, desde 1000-01-01 00:00:00 hasta 9999-12-31 23:59:59.
2. Simplicidad: DATETIME es más fácil de entender y utilizar que TIMESTAMP, especialmente para aquellos que no están familiarizados con la zona horaria UTC.
3. Compatibilidad: DATETIME es compatible con la mayoría de las aplicaciones y frameworks, mientras que TIMESTAMP puede requerir configuraciones adicionales.
Sin embargo, TIMESTAMP es muy útil en ciertas situaciones, como:
1. Almacenamiento de fechas y horas con zona horaria: TIMESTAMP es ideal para almacenar fechas y horas con zona horaria, especialmente cuando se trabaja con aplicaciones globales.
2. Actualización automática: TIMESTAMP se puede configurar para actualizarse automáticamente cuando se inserta o actualiza un registro.

En resumen, aunque DATETIME es el tipo de dato más utilizado en MySQL, TIMESTAMP es muy útil en ciertas situaciones y puede ser la mejor opción dependiendo de las necesidades específicas de tu proyecto.


In [9]:
# Conversion de zona horaria */
# Cuando en londres era 2004-01-01 12:00:00, en Guadalajara era 2004-01-01 06:00:00, ya que son 6 horas hacia atras */
# La funcion Convert_TZ requiere los argumentos: date time, zona horaria en que se registro el dato, zona horaria deseada */

query="""
SELECT 	CONVERT_TZ("2004-01-01 12:00:00","+00:00","-06:00") as HorarioCST; 
"""
df = pd.read_sql(query, engine)
df.head(10)

,HorarioCST
0,2004-01-01 06:00:00


In [10]:
# Conversion de zona horaria */
# Cuando en londres era 2004-01-01 12:00:00, en San Francisco California era 2004-01-01 06:00:00, ya que son 8 horas hacia atras */

query="""
SELECT 	CONVERT_TZ("2004-01-01 12:00:00","+00:00","-08:00") as HorarioCST;
"""
df = pd.read_sql(query, engine)
df.head(10)

,HorarioCST
0,2004-01-01 04:00:00


In [11]:
# /* Conversion de zona horaria de una columna */

query="""
SELECT
    ventasgerentes_id,
    fecha,
    convert_tz(fecha,"+00:00","-6:00") as Horario_Guadalajara
from
    preparacion.ventasgerentes;
"""
df = pd.read_sql(query, engine)
df.head(10)

,ventasgerentes_id,fecha,Horario_Guadalajara
0,1,2012-09-01,2012-08-31 18:00:00
1,2,2012-09-03,2012-09-02 18:00:00
2,3,2012-09-03,2012-09-02 18:00:00
3,4,2012-09-05,2012-09-04 18:00:00
4,5,2012-09-06,2012-09-05 18:00:00
5,6,2012-09-06,2012-09-05 18:00:00
6,7,2012-09-07,2012-09-06 18:00:00
7,8,2012-09-10,2012-09-09 18:00:00
8,9,2012-09-10,2012-09-09 18:00:00
9,10,2012-09-11,2012-09-10 18:00:00


##### 3-4 Fechas y Horas

In [ ]:

# TimeStamp: Estampa de tiempo */
# MYSQL maneja las fechas en formato YYYY-MM-DD
# Mexico usa las fechas en formato dd-mm-aaaa
# EU usa las fechas en formato mm-dd-aaaa

# Veremos 3 casos: Cambiar el formato de las fechas, extraer partes de la fecha y armar una fecha a partir de sus partes. */


In [13]:
# Obtener fecha actual

query="""
select current_date();
"""
df = pd.read_sql(query, engine)
df.head(10)

,current_date()
0,2026-09-24


In [14]:
# Obtener fecha y hora actual

query="""
select now();
"""
df = pd.read_sql(query, engine)
df.head(10)

,now()
0,2026-09-24 18:20:13


In [15]:
# date_format() es una función de MySQL que permite mostrar una fecha o un valor de fecha y hora en un formato específico.
# Recibe dos argumentos: la fecha a formatear y el formato deseado.
# El primer argumento es la fecha a formatear, que puede ser una fecha literal, una columna de tipo fecha o un valor de fecha y hora. 
# Generalmente MySQL reconoce automaticamente el primer argumento como una fecha, pero si no lo hace, podemos tranformar el primer argumento a un tipo de dato fecha/hora usando la función STR_TO_DATE() que veremos más adelante.
# Retorna un valor de tipo cadena de texto (string) con la fecha formateada según el segundo argumento.

# Código	y su significado
# %Y	Año con 4 dígitos
# %m	Mes con 2 dígitos
# %d	Día con 2 dígitos
# %H	Hora en formato 24 horas
# %h	Hora en formato 12 horas
# %i	Minutos
# %s	Segundos

# En resumen
# STR_TO_DATE(): Recibe texto, nosotros le indicamos como interpretar la cadena y retorna un tipo de dato fecha/hora
# DATE_FORMAT(): Recibe un tipo de dato fecha/hora y retorna texto en el formato de fecha/hora que le indiquemos

# Mostrar una fecha en formato Año-Mes-Día
query="""
select date_format("2020-12-17","%Y-%m-%d") as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha
0,2020-12-17


In [16]:
# Convertir una cadena de texto a tipo de dato fecha/hora

query="""
select str_to_date('17/12/2020 14:15:16','%d/%m/%Y %H:%i:%s') as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head()

,fecha
0,2020-12-17 14:15:16


In [17]:
# Mostrar una fecha en formato Año-Mes-Día

query="""
select date_format("2020-12-17","%y-%M-%D");
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,"date_format(""2020-12-17"",""%y-%M-%D"")"
0,20-December-17th


In [18]:
# Mostrar una fecha en formato Año-Mes-Día con Día=01

query="""
select date_format("2020-12-17","%Y-%m-01");
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,"date_format(""2020-12-17"",""%Y-%m-01"")"
0,2020-12-01


In [19]:
# Convertir un datetime a date

query="""
select date("2020-12-17 14:00:00") as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha
0,2020-12-17


In [20]:
#  Extraer el dia, mes, año, hora, minuto, segundo de un datetime

query="""
select
    "2020-12-17 14:15:20" as fecha,
    extract(day from "2020-12-17 14:15:20") 	as dia,
    extract(month from "2020-12-17 14:15:20") 	as mes,
    extract(year from "2020-12-17 14:15:20") 	as año,
    extract(hour from "2020-12-17 14:15:20") 	as hora,
    extract(minute from "2020-12-17 14:15:20") 	as minutos,
    extract(second from "2020-12-17 14:15:20") 	as segundos;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha,dia,mes,año,hora,minutos,segundos
0,2020-12-17 14:15:20,17,12,2020,14,15,20


In [21]:
# Imprimir una fecha y hora a partir de cadenas de texto

query="""
select date_format(concat("2020-12-17"," ","14:15:16"),"%Y-%m-%d %h:%m:%s") as fecha_hora; 
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha_hora
0,2020-12-17 02:12:16


In [22]:
# Crear una fecha a partir de cadenas de texto 

quer="""
select date_format(concat("2020-12-17"),"%Y-%m-%d") as fecha; 
"""
df = pd.read_sql(text(quer), engine)
df.head(10)

,fecha
0,2020-12-17


In [23]:
# Crear fecha a partir del año y cuantos días han transcurrido

query="""
select makedate(2020,352) as fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha
0,2020-12-17


In [24]:
# Crear una hora a partir de hora, minuto y segundos

query="""
select maketime(14,15,20);
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,"maketime(14,15,20)"
0,0 days 14:15:20


##### 3-5-Date Math

In [25]:
# Calcular la diferencia entre 2 fechas y expresarla en días, meses, años, horas, minutos y segundos

query="""
select
    timestampdiff(DAY, "2020-01-01","2020-02-01") 	as diferencia_en_dias,
    timestampdiff(MONTH, "2020-01-01","2020-02-01") as diferencia_en_meses,
    timestampdiff(YEAR, "2020-01-01","2020-02-01") 	as diferencia_en_años,
    timestampdiff(HOUR, "2020-01-01","2020-02-01") 	as diferencia_en_horas,
    timestampdiff(MINUTE, "2020-01-01","2020-02-01") as diferencia_en_minutos,
    timestampdiff(SECOND, "2020-01-01","2020-02-01") as diferencia_en_segundos;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,diferencia_en_dias,diferencia_en_meses,diferencia_en_años,diferencia_en_horas,diferencia_en_minutos,diferencia_en_segundos
0,31,1,0,744,44640,2678400


In [26]:
# Obtener una fecha a partir de una fecha dada y un intervalo

query="""
SELECT DATE_ADD("2017-12-01", interval 16 day ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2017-12-17


In [27]:
# Obtener una fecha a partir de una fecha dada y un intervalo

query="""
SELECT DATE_ADD("2017-12-01", interval 1 month ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2018-01-01


In [28]:
# Obtener una fecha a partir de una fecha dada y un intervalo */

query="""
SELECT DATE_ADD("2017-12-01", interval 5 second ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)


,nueva_fecha
0,2017-12-01 00:00:05


In [29]:
# Obtener una fecha a partir de una fecha dada y un intervalo */

query="""
SELECT DATE_ADD("2017-12-01", interval -1 day ) as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2017-11-30


In [30]:
# Es posible sumar fechas y multiples intervalos sin usar la funcion date_add */

query="""
select date("2020-01-01") + interval 1 day + interval 1 hour - interval 15 minute + interval 15 second as nueva_fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_fecha
0,2020-01-02 00:45:15


In [31]:
# Podemos usar funciones de fecha al filtrar datos

query="""
select
    fecha,
    sexo 
from
    preparacion.ventasgerentes
where fecha > date("2013-01-01") + interval 30 day
order by fecha asc;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha,sexo
0,2013-02-01,M
1,2013-02-02,H
2,2013-02-02,H
3,2013-02-02,H
4,2013-02-03,M
5,2013-02-06,H
6,2013-02-06,H
7,2013-02-06,H
8,2013-02-06,M
9,2013-02-07,H


##### 3-6-Time Math

In [32]:
# Sumar horas, minutos y segundos a un tiempo determinado

query="""
select time("05:00:00") + interval 1 hour + interval 5 minute + interval 15 second as nueva_hora;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nueva_hora
0,0 days 06:05:15


In [33]:
# Diferencia entre 2 tiempos */

query="""
select timediff("10:00:00","05:00:00") as diferencia;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,diferencia
0,0 days 05:00:00


##### 3-7-Tendencias Básicas

In [34]:
# Todos los registros de la tabla "retail_sales" 

query="""
select
    *
from
    serie_de_tiempo.retail_sales
order by 
    sales_month asc;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,sales_id,sales_month,naics_code,kind_of_business,reason_for_null,sales
0,1,1992-01-01,441,Motor vehicle and parts dealers,NaN,29811
1,2,1992-01-01,4411,Automobile dealers,NaN,25800
2,3,1992-01-01,"4411, 4412",Automobile and other motor vehicle dealers,NaN,26788
3,4,1992-01-01,44111,New car dealers,NaN,24056
4,5,1992-01-01,44112,Used car dealers,NaN,1744
5,6,1992-01-01,4413,"Automotive parts, acc., and tire stores",NaN,3023
6,7,1992-01-01,442,Furniture and home furnishings stores,NaN,3846
7,8,1992-01-01,"442, 443","Furniture, home furn, electronics, and applian...",NaN,7503
8,9,1992-01-01,4421,Furniture stores,NaN,2392
9,10,1992-01-01,4422,Home furnishings stores,NaN,1454


In [38]:
# Todos los registros de la tabla "tamaños"

query="""
select
    *
from
    serie_de_tiempo.tamaños;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,tamaños_id,Tamaño,Descripcion,Tamaño_en_CMS
0,1,P,Pequeño,20
1,2,M,Mediano,30
2,3,G,Grande,40
3,4,EG,Extra Grande,50


In [39]:
# Ventas totales por mes

query = """
select
    sales_month,
    sales
from
    serie_de_tiempo.retail_sales
where 
    kind_of_business IN ("Retail and food services sales, total")
order by 
    sales_month asc;
"""
df = pd.read_sql(text(query), engine)
df.head(10)


,sales_month,sales
0,1992-01-01,146376
1,1992-02-01,147079
2,1992-03-01,159336
3,1992-04-01,163669
4,1992-05-01,170068
5,1992-06-01,168663
6,1992-07-01,169890
7,1992-08-01,170364
8,1992-09-01,164617
9,1992-10-01,173655


In [41]:
# Ventas totales por año

query="""
select
    extract(year from sales_month) as ventas_anuales,
    sum(sales)
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Retail and food services sales, total")
group by
    ventas_anuales
order by
    ventas_anuales asc;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,ventas_anuales,sum(sales)
0,1992,2014102.0
1,1993,2153095.0
2,1994,2330235.0
3,1995,2450628.0
4,1996,2603794.0
5,1997,2726131.0
6,1998,2852956.0
7,1999,3086990.0
8,2000,3287537.0
9,2001,3378906.0


##### 3-8-Tendencias Compuestas

In [49]:

query = """
select
    extract(year from sales_month) as año,
    kind_of_business,
    sum(sales) ventas
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ('Book stores','Sporting goods stores', 'Hobby, toy, and game stores')
group by
    año, kind_of_business;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,año,kind_of_business,ventas
0,1992,Sporting goods stores,15583.0
1,1992,"Hobby, toy, and game stores",11251.0
2,1992,Book stores,8327.0
3,1993,Sporting goods stores,16791.0
4,1993,"Hobby, toy, and game stores",11651.0
5,1993,Book stores,9108.0
6,1994,Sporting goods stores,18825.0
7,1994,"Hobby, toy, and game stores",12850.0
8,1994,Book stores,10107.0
9,1995,Sporting goods stores,19869.0


##### 3-9-Tendencias Complejas con porcentajes

In [59]:
# Ventas totales agrupado por fecha y tipo de negocio

query = """
select
    date(sales_month) as mes_venta,
    kind_of_business,
    sales
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,mes_venta,kind_of_business,sales
0,1992-01-01,Men's clothing stores,701
1,1992-01-01,Women's clothing stores,1873
2,1992-02-01,Men's clothing stores,658
3,1992-02-01,Women's clothing stores,1991
4,1992-03-01,Men's clothing stores,731
5,1992-03-01,Women's clothing stores,2403
6,1992-04-01,Men's clothing stores,816
7,1992-04-01,Women's clothing stores,2665
8,1992-05-01,Men's clothing stores,856
9,1992-05-01,Women's clothing stores,2752


In [61]:
# Venta totales agrupado por año y tipo de negocio

query="""
select
    extract(year from sales_month) as año,
    kind_of_business,
    sum(sales)
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
group by
    año,kind_of_business;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,año,kind_of_business,sum(sales)
0,1992,Men's clothing stores,10179.0
1,1992,Women's clothing stores,31815.0
2,1993,Men's clothing stores,9962.0
3,1993,Women's clothing stores,32350.0
4,1994,Men's clothing stores,10032.0
5,1994,Women's clothing stores,30585.0
6,1995,Men's clothing stores,9315.0
7,1995,Women's clothing stores,28696.0
8,1996,Men's clothing stores,9546.0
9,1996,Women's clothing stores,28238.0


In [64]:
# Venta totales agrupado por año y tipo de negocio

query = """
select
    extract(year from sales_month) as año,
    kind_of_business,
    sum(sales)
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
group by
    año,kind_of_business;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,año,kind_of_business,sum(sales)
0,1992,Men's clothing stores,10179.0
1,1992,Women's clothing stores,31815.0
2,1993,Men's clothing stores,9962.0
3,1993,Women's clothing stores,32350.0
4,1994,Men's clothing stores,10032.0
5,1994,Women's clothing stores,30585.0
6,1995,Men's clothing stores,9315.0
7,1995,Women's clothing stores,28696.0
8,1996,Men's clothing stores,9546.0
9,1996,Women's clothing stores,28238.0


In [67]:
# Diferencia de ventas totales entre hombres y mujeres agrupado por año

query="""
WITH T as (
select
    extract(year from sales_month) as año,
    sum(case when kind_of_business = "Women\'s clothing stores" then sales else 0 end) as venta_mujeres,
    sum(case when kind_of_business = "Men\'s clothing stores"   then sales else 0 end) as venta_hombres
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
group by
    año
	)
select 
    año,
    venta_mujeres,
    venta_hombres,
    venta_mujeres - venta_hombres as mujeres_menos_hombres
from T;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,año,venta_mujeres,venta_hombres,mujeres_menos_hombres
0,1992,31815.0,10179.0,21636.0
1,1993,32350.0,9962.0,22388.0
2,1994,30585.0,10032.0,20553.0
3,1995,28696.0,9315.0,19381.0
4,1996,28238.0,9546.0,18692.0
5,1997,27822.0,10069.0,17753.0
6,1998,28332.0,10196.0,18136.0
7,1999,29549.0,9667.0,19882.0
8,2000,31447.0,9507.0,21940.0
9,2001,31453.0,8625.0,22828.0


In [70]:
# Tasa de ventas totales de mujeres respecto de ventas de hombres agrupado por año

query="""
WITH T as (
select
    extract(year from sales_month) as año,
    sum(case when kind_of_business = "Women\'s clothing stores" then sales else 0 end) as venta_mujeres,
    sum(case when kind_of_business = "Men\'s clothing stores"   then sales else 0 end) as venta_hombres
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
group by
    año
)
select 
    año, 
    venta_mujeres,
    venta_hombres,
    venta_mujeres / venta_hombres as mujeres_entre_hombres
from T;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,año,venta_mujeres,venta_hombres,mujeres_entre_hombres
0,1992,31815.0,10179.0,3.1256
1,1993,32350.0,9962.0,3.2473
2,1994,30585.0,10032.0,3.0487
3,1995,28696.0,9315.0,3.0806
4,1996,28238.0,9546.0,2.9581
5,1997,27822.0,10069.0,2.7631
6,1998,28332.0,10196.0,2.7787
7,1999,29549.0,9667.0,3.0567
8,2000,31447.0,9507.0,3.3078
9,2001,31453.0,8625.0,3.6467


##### 3-10-Porcentaje del total

In [ ]:
# Se pueden usar 2 metodos distintos: Self Join (unir una tabla consigo misma) y con Funciones de Ventana (Over)

# Metodo 1: SELF JOIN
# Obtener el porcentaje de ventas de hombres y mujeres respecto del total de ventas mensuales de ambos sexos

query="""
select
    a.sales_month, 
    a.kind_of_business, 
    a.sales, 
    b.ventas_totales_mes, 
    100 * a.sales / b.ventas_totales_mes as pct_ventas_totales
from 
    (                                                              #De este subquery obtenemos las ventas desglosadas por sexo
    select
        sales_month,
        kind_of_business,
        sales
    from
        serie_de_tiempo.retail_sales as a
    where 
        kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
    ) as a
    JOIN
    (                                                              #De este subquery obtenemos las ventas sin desglosar
    select
        sales_month,
        sum(sales) as ventas_totales_mes
    from
        serie_de_tiempo.retail_sales 
    where
        kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
        group by sales_month
    ) as b
    ON a.sales_month=b.sales_month;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,sales_month,kind_of_business,sales,ventas_totales_mes,pct_ventas_totales
0,1992-01-01,Men's clothing stores,701,2574.0,27.2339
1,1992-01-01,Women's clothing stores,1873,2574.0,72.7661
2,1992-02-01,Men's clothing stores,658,2649.0,24.8396
3,1992-02-01,Women's clothing stores,1991,2649.0,75.1604
4,1992-03-01,Men's clothing stores,731,3134.0,23.3248
5,1992-03-01,Women's clothing stores,2403,3134.0,76.6752
6,1992-04-01,Men's clothing stores,816,3481.0,23.4415
7,1992-04-01,Women's clothing stores,2665,3481.0,76.5585
8,1992-05-01,Men's clothing stores,856,3608.0,23.7251
9,1992-05-01,Women's clothing stores,2752,3608.0,76.2749


In [79]:
# Método 2: FUNCION VENTANA
# Obtener el porcentaje de ventas de hombres y mujeres respecto del total de ventas mensuales de ambos sexos

query="""
select
    sales_month,
    kind_of_business,
    sales,
    sum(sales) over (partition by sales_month) as ventas_totales_mes,             # Agregar a cada fila la venta total del mes
    100*sales/(sum(sales) over (partition by sales_month)) as pct_ventas_totales  # Calcula el porcentaje respecto del total de ventas del mes
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores");
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,sales_month,kind_of_business,sales,ventas_totales_mes,pct_ventas_totales
0,1992-01-01,Men's clothing stores,701,2574.0,27.2339
1,1992-01-01,Women's clothing stores,1873,2574.0,72.7661
2,1992-02-01,Men's clothing stores,658,2649.0,24.8396
3,1992-02-01,Women's clothing stores,1991,2649.0,75.1604
4,1992-03-01,Men's clothing stores,731,3134.0,23.3248
5,1992-03-01,Women's clothing stores,2403,3134.0,76.6752
6,1992-04-01,Men's clothing stores,816,3481.0,23.4415
7,1992-04-01,Women's clothing stores,2665,3481.0,76.5585
8,1992-05-01,Men's clothing stores,856,3608.0,23.7251
9,1992-05-01,Women's clothing stores,2752,3608.0,76.2749


In [88]:
# Ahora obtener el porcentaje que cada mes representa de las ventas anuales
# Metodo 1: SELF JOIN

query="""
select
    a.anio, 
    a.sales_month, 
    a.kind_of_business, 
    a.sales,  
    b.ventas_anuales, 
    100 * a.sales / b.ventas_anuales as pct_ventas
from
    (
    select
        extract(year from sales_month) as anio,
        sales_month, 
        kind_of_business, 
        sales
    from
        serie_de_tiempo.retail_sales
    where
        kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
    ) as a 
    JOIN
    (
    select
        extract(year from sales_month) as anio,
        kind_of_business,
        sum(sales) as ventas_anuales
    from
        serie_de_tiempo.retail_sales
    where
        kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
    group by 
        anio,kind_of_business
    ) as b
    ON a.anio = b.anio AND a.kind_of_business=b.kind_of_business
order by
    1,3,2
    ;
"""
df = pd.read_sql(text(query),engine)
df.head(24)

,anio,sales_month,kind_of_business,sales,ventas_anuales,pct_ventas
0,1992,1992-01-01,Men's clothing stores,701,10179.0,6.8867
1,1992,1992-02-01,Men's clothing stores,658,10179.0,6.4643
2,1992,1992-03-01,Men's clothing stores,731,10179.0,7.1815
3,1992,1992-04-01,Men's clothing stores,816,10179.0,8.0165
4,1992,1992-05-01,Men's clothing stores,856,10179.0,8.4095
5,1992,1992-06-01,Men's clothing stores,853,10179.0,8.3800
6,1992,1992-07-01,Men's clothing stores,714,10179.0,7.0144
7,1992,1992-08-01,Men's clothing stores,777,10179.0,7.6334
8,1992,1992-09-01,Men's clothing stores,762,10179.0,7.4860
9,1992,1992-10-01,Men's clothing stores,841,10179.0,8.2621


In [89]:
# Metodo 2: FUNCION VENTANA

query="""
select
    extract(year from sales_month) as year_extract,
    sales_month,
    kind_of_business,
    sales,
    sum(sales) over (partition by extract(year from sales_month),kind_of_business) as venta_anual,
    100 * sales / sum(sales) over (partition by extract(year from sales_month),kind_of_business) as pct_ventas
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
order by 
    1,3,2
    ;
"""
df = pd.read_sql(text(query),engine)
df.head(24)


,year_extract,sales_month,kind_of_business,sales,venta_anual,pct_ventas
0,1992,1992-01-01,Men's clothing stores,701,10179.0,6.8867
1,1992,1992-02-01,Men's clothing stores,658,10179.0,6.4643
2,1992,1992-03-01,Men's clothing stores,731,10179.0,7.1815
3,1992,1992-04-01,Men's clothing stores,816,10179.0,8.0165
4,1992,1992-05-01,Men's clothing stores,856,10179.0,8.4095
5,1992,1992-06-01,Men's clothing stores,853,10179.0,8.3800
6,1992,1992-07-01,Men's clothing stores,714,10179.0,7.0144
7,1992,1992-08-01,Men's clothing stores,777,10179.0,7.6334
8,1992,1992-09-01,Men's clothing stores,762,10179.0,7.4860
9,1992,1992-10-01,Men's clothing stores,841,10179.0,8.2621


##### 3-11-Cambio Porcentual a lo largo del tiempo

In [100]:

query="""
select
    anio_venta,
    kind_of_business,
    ventas,
    first_value(ventas) over (partition by kind_of_business order by anio_venta) as primer_venta,
    ((ventas / first_value(ventas) over (partition by kind_of_business order by anio_venta)) - 1) * 100 as indice_ventas
from 
    (
    select
        extract(year from sales_month) as anio_venta, 
        kind_of_business,
        sum(sales) as ventas
    from
        serie_de_tiempo.retail_sales
    where
        kind_of_business IN ("Men\'s clothing stores","Women\'s clothing stores")
    group by
        anio_venta,kind_of_business
    ) as A;
"""
df = pd.read_sql(text(query),engine)
df.head(20)

,anio_venta,kind_of_business,ventas,primer_venta,indice_ventas
0,1992,Men's clothing stores,10179.0,10179.0,0.00
1,1993,Men's clothing stores,9962.0,10179.0,-2.13
2,1994,Men's clothing stores,10032.0,10179.0,-1.44
3,1995,Men's clothing stores,9315.0,10179.0,-8.49
4,1996,Men's clothing stores,9546.0,10179.0,-6.22
5,1997,Men's clothing stores,10069.0,10179.0,-1.08
6,1998,Men's clothing stores,10196.0,10179.0,0.17
7,1999,Men's clothing stores,9667.0,10179.0,-5.03
8,2000,Men's clothing stores,9507.0,10179.0,-6.60
9,2001,Men's clothing stores,8625.0,10179.0,-15.27


##### 3-12-Introduccion a Ventanas Moviles

##### 3-13- Ventanas Moviles

In [110]:
# Hay 2 maneras de hacerlo, usando self join y usando las funciones de ventana over

# Método con self join

query="""
select
    a.sales_month, 
    a.sales, 
    /*b.sales_month as ventana_movil,*/ 
    AVG(b.sales) as promedio_de_ventas_ventana_movil,
    count(*) as conteo_de_filas
from
    serie_de_tiempo.retail_sales as a 
    join
    serie_de_tiempo.retail_sales as b 
    on a.kind_of_business = b.kind_of_business and b.sales_month between a.sales_month - interval 11 month and a.sales_month
where
    a.kind_of_business IN ("Women\'s clothing stores")
group by 1,2
order by 1;
"""
df = pd.read_sql(text(query),engine)
df.head(20)

,sales_month,sales,promedio_de_ventas_ventana_movil,conteo_de_filas
0,1992-01-01,1873,1873.0000,1
1,1992-02-01,1991,1932.0000,2
2,1992-03-01,2403,2089.0000,3
3,1992-04-01,2665,2233.0000,4
4,1992-05-01,2752,2336.8000,5
5,1992-06-01,2424,2351.3333,6
6,1992-07-01,2373,2354.4286,7
7,1992-08-01,2657,2392.2500,8
8,1992-09-01,2560,2410.8889,9
9,1992-10-01,2755,2445.3000,10


In [115]:
# Metodo 2: Funciones de ventana

# Metodo con funciones de ventana OVER PARTITION

query="""
select
    sales_month,
    sales,
    avg(sales) 	over (order by sales_month rows between  11 preceding and current row) 	as promedio_movil,
    count(sales) 	over (order by sales_month rows between  11 preceding and current row)	as conteo_de_filas
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Women\'s clothing stores");
"""
df = pd.read_sql(text(query),engine)
df.head(20)

,sales_month,sales,promedio_movil,conteo_de_filas
0,1992-01-01,1873,1873.0000,1
1,1992-02-01,1991,1932.0000,2
2,1992-03-01,2403,2089.0000,3
3,1992-04-01,2665,2233.0000,4
4,1992-05-01,2752,2336.8000,5
5,1992-06-01,2424,2351.3333,6
6,1992-07-01,2373,2354.4286,7
7,1992-08-01,2657,2392.2500,8
8,1992-09-01,2560,2410.8889,9
9,1992-10-01,2755,2445.3000,10


##### 3-14-Acumulado YTD

In [122]:
# Funciones acumulativas
# Obtener las ventas acumuladas y que se reinicie el acumulado cada 3 meses

query="""
select
    sales_month,
    sales as ventas_del_mes_actual,
    sum(sales) over (partition by extract(year from sales_month), extract(quarter from sales_month) order by sales_month) as ventas_acumuladas
from
    serie_de_tiempo.retail_sales
where
    kind_of_business IN ("Women\'s clothing stores");
"""
df = pd.read_sql(text(query),engine)
df.head(24)

,sales_month,ventas_del_mes_actual,ventas_acumuladas
0,1992-01-01,1873,1873.0
1,1992-02-01,1991,3864.0
2,1992-03-01,2403,6267.0
3,1992-04-01,2665,2665.0
4,1992-05-01,2752,5417.0
5,1992-06-01,2424,7841.0
6,1992-07-01,2373,2373.0
7,1992-08-01,2657,5030.0
8,1992-09-01,2560,7590.0
9,1992-10-01,2755,2755.0


##### 3-15-Estacionalidad

##### 3-16-Periodo vs Periodo

In [ ]:
# Año Actual vs Año Pasado, Mes Actual vs Mes Pasado, Día Actual vs Día Anterior, etc
# Se puede utilizar la función LAG con valores enteros postivos para obtener valores anteriores, y con valores negativos se obtienen valores posteriores

# Se puede hacer de 2 maneras distintas: Self Join y función Ventana

# Comparar ventas de mes actual vs mes anterior
# Nota: Para esta operacion MySQL no esta realmente calculando el mes previo en terminos de fechas, sino que dada la particion y el orden elegido en la clausula over, toma el valor de la fila inmediata anterior

query = """
select
    kind_of_business,
    sales_month,
    sales as ventas,
    lag(sales_month) 	over (partition by kind_of_business order by sales_month) as mes_previo,
    lag(sales) 		over (partition by kind_of_business order by sales_month) as ventas_mes_previo,
    ((sales/lag(sales) over (partition by kind_of_business order by sales_month))-1)*100 as porcentaje_crecimiento
from
    serie_de_tiempo.retail_sales
where
    kind_of_business = "Book Stores";
"""
df = pd.read_sql(text(query),engine)
df.head(20)

,kind_of_business,sales_month,ventas,mes_previo,ventas_mes_previo,porcentaje_crecimiento
0,Book stores,1992-01-01,790,NaT,NaN,NaN
1,Book stores,1992-02-01,539,1992-01-01,790.0,-31.77
2,Book stores,1992-03-01,535,1992-02-01,539.0,-0.74
3,Book stores,1992-04-01,523,1992-03-01,535.0,-2.24
4,Book stores,1992-05-01,552,1992-04-01,523.0,5.54
5,Book stores,1992-06-01,589,1992-05-01,552.0,6.70
6,Book stores,1992-07-01,592,1992-06-01,589.0,0.51
7,Book stores,1992-08-01,894,1992-07-01,592.0,51.01
8,Book stores,1992-09-01,861,1992-08-01,894.0,-3.69
9,Book stores,1992-10-01,645,1992-09-01,861.0,-25.09


In [ ]:
# Comparar ventas de año actual vs año anterior */

query="""
select
    anio,
    ventas_anuales,
    lag(anio) over (order by anio) as anio_anterior,
    lag(ventas_anuales) over (order by anio) as ventas_anio_anterior,
    ((ventas_anuales/lag(ventas_anuales) over (order by anio))-1)*100 as porcentaje_crecimiento
from 
    (
    select
        extract(year from sales_month) as anio,
        sum(sales) as ventas_anuales 
    from
        serie_de_tiempo.retail_sales
    where
        kind_of_business = "Book Stores"
    group by
        1
    ) as a;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,anio,ventas_anuales,anio_anterior,ventas_anio_anterior,porcentaje_crecimiento
0,1992,8327.0,NaN,NaN,NaN
1,1993,9108.0,1992.0,8327.0,9.38
2,1994,10107.0,1993.0,9108.0,10.97
3,1995,11196.0,1994.0,10107.0,10.77
4,1996,11905.0,1995.0,11196.0,6.33
5,1997,12742.0,1996.0,11905.0,7.03
6,1998,13282.0,1997.0,12742.0,4.24
7,1999,14172.0,1998.0,13282.0,6.70
8,2000,14879.0,1999.0,14172.0,4.99
9,2001,15098.0,2000.0,14879.0,1.47


##### 3-17-Mismo Mes Año Pasado

In [132]:
# Comparar cada mes con el mismo mes 1 año hacia atrás

query="""
select
    sales_month,
    sales,
    extract(month from sales_month) as mes,
    lag(sales_month) 	over (partition by extract(month from sales_month) order by sales_month) as mes_año_previo,
    lag(sales) 		over (partition by extract(month from sales_month) order by sales_month) as ventas_mes_año_previo,
    (sales - lag(sales)	over (partition by extract(month from sales_month) order by sales_month)) as diferencia_absoluta,
    ((sales / lag(sales) 	over (partition by extract(month from sales_month) order by sales_month)) -1)*100 as porcentaje_crecimiento
from
    serie_de_tiempo.retail_sales
where
    kind_of_business = "Book Stores";
"""
df = pd.read_sql(text(query),engine)
df.head(20)

,sales_month,sales,mes,mes_año_previo,ventas_mes_año_previo,diferencia_absoluta,porcentaje_crecimiento
0,1992-01-01,790,1,NaT,NaN,NaN,NaN
1,1993-01-01,998,1,1992-01-01,790.0,208.0,26.33
2,1994-01-01,1053,1,1993-01-01,998.0,55.0,5.51
3,1995-01-01,1308,1,1994-01-01,1053.0,255.0,24.22
4,1996-01-01,1373,1,1995-01-01,1308.0,65.0,4.97
5,1997-01-01,1558,1,1996-01-01,1373.0,185.0,13.47
6,1998-01-01,1463,1,1997-01-01,1558.0,-95.0,-6.10
7,1999-01-01,1514,1,1998-01-01,1463.0,51.0,3.49
8,2000-01-01,1505,1,1999-01-01,1514.0,-9.0,-0.59
9,2001-01-01,1585,1,2000-01-01,1505.0,80.0,5.32


##### 3-18-Varios Periodos Previos y Conclusion

In [142]:
# Es posible obtener los datos con 1,2,3,...,n periodos de retraso.
# Para cada mes obtener las ventas del mismo mes pero de 1,2 y 3 años anteriores.

query="""
select
    sales_month,
    sales,
    lag(sales,1) over (partition by extract(month from sales_month) order by sales_month) as ventas_lag_1_año,
    lag(sales,2) over (partition by extract(month from sales_month) order by sales_month) as ventas_lag_2_año,
    lag(sales,3) over (partition by extract(month from sales_month) order by sales_month) as ventas_lag_3_año
from
    serie_de_tiempo.retail_sales
where
    kind_of_business = "Book Stores";
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,sales_month,sales,ventas_lag_1_año,ventas_lag_2_año,ventas_lag_3_año
0,1992-01-01,790,NaN,NaN,NaN
1,1993-01-01,998,790.0,NaN,NaN
2,1994-01-01,1053,998.0,790.0,NaN
3,1995-01-01,1308,1053.0,998.0,790.0
4,1996-01-01,1373,1308.0,1053.0,998.0
5,1997-01-01,1558,1373.0,1308.0,1053.0
6,1998-01-01,1463,1558.0,1373.0,1308.0
7,1999-01-01,1514,1463.0,1558.0,1373.0
8,2000-01-01,1505,1514.0,1463.0,1558.0
9,2001-01-01,1585,1505.0,1514.0,1463.0


In [143]:
df.tail(10)

,sales_month,sales,ventas_lag_1_año,ventas_lag_2_año,ventas_lag_3_año
338,2011-12-01,1400,1879.0,1911.0,2008.0
339,2012-12-01,1318,1400.0,1879.0,1911.0
340,2013-12-01,1327,1318.0,1400.0,1879.0
341,2014-12-01,1332,1327.0,1318.0,1400.0
342,2015-12-01,1321,1332.0,1327.0,1318.0
343,2016-12-01,1249,1321.0,1332.0,1327.0
344,2017-12-01,1114,1249.0,1321.0,1332.0
345,2018-12-01,1122,1114.0,1249.0,1321.0
346,2019-12-01,1037,1122.0,1114.0,1249.0
347,2020-12-01,900,1037.0,1122.0,1114.0


In [148]:
# Obtener el % de crecimiento del mes con respecto al promedio de venta del mismo mes pero de hace 1 año, 2 años y 3 años */

query="""
select
    sales_month,
    sales,
    avg(sales) over (partition by extract(month from sales_month) order by sales_month rows between 3 preceding and 1 preceding) as promedio_venta_3_periodos_antes,
    (sales / avg(sales) over (partition by extract(month from sales_month) order by sales_month rows between 3 preceding and 1 preceding)) as procentaje_crecimiento
from
    serie_de_tiempo.retail_sales
where
    kind_of_business = "Book Stores";
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,sales_month,sales,promedio_venta_3_periodos_antes,procentaje_crecimiento
0,1992-01-01,790,NaN,NaN
1,1993-01-01,998,790.0000,1.2633
2,1994-01-01,1053,894.0000,1.1779
3,1995-01-01,1308,947.0000,1.3812
4,1996-01-01,1373,1119.6667,1.2263
5,1997-01-01,1558,1244.6667,1.2517
6,1998-01-01,1463,1413.0000,1.0354
7,1999-01-01,1514,1464.6667,1.0337
8,2000-01-01,1505,1511.6667,0.9956
9,2001-01-01,1585,1494.0000,1.0609


In [149]:
df.tail(10)

,sales_month,sales,promedio_venta_3_periodos_antes,procentaje_crecimiento
338,2011-12-01,1400,1932.6667,0.7244
339,2012-12-01,1318,1730.0000,0.7618
340,2013-12-01,1327,1532.3333,0.8660
341,2014-12-01,1332,1348.3333,0.9879
342,2015-12-01,1321,1325.6667,0.9965
343,2016-12-01,1249,1326.6667,0.9415
344,2017-12-01,1114,1300.6667,0.8565
345,2018-12-01,1122,1228.0000,0.9137
346,2019-12-01,1037,1161.6667,0.8927
347,2020-12-01,900,1091.0000,0.8249
